# LexAI v3.2 — Phase 4: GPU Pipeline

**What this notebook does:**
1. Batch-encode all 500 cases with LegalBERT (GPU)
2. Build FAISS index
3. HDBSCAN vs KMeans clustering (silhouette picks winner)
4. UMAP 2D coordinates for visualization
5. BERTopic cluster labels
6. Verdict inconsistency gap detection

**Outputs:** 7 files saved to `data/processed/` and Google Drive

**Recovery:** Every cell saves a checkpoint. If Colab disconnects, re-run from the failed cell.

In [ ]:
# Cell 1 — GPU check
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")
    print("    Go to: Runtime > Change runtime type > T4 GPU")
    print("    Then:  Runtime > Restart and run all")

In [ ]:
# Cell 2 — Mount Drive + checkpoint tracker
from google.colab import drive
drive.mount('/content/drive')
import os, json
from pathlib import Path

GDRIVE = '/content/drive/MyDrive/lexai_outputs'
os.makedirs(GDRIVE, exist_ok=True)

CHECKPOINT_FILE = f"{GDRIVE}/checkpoint.json"

def load_checkpoints():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}

def save_checkpoint(name):
    cp = load_checkpoints()
    cp[name] = True
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(cp, f, indent=2)
    print(f"Checkpoint saved: {name}")

checkpoints = load_checkpoints()
print(f"Outputs folder: {GDRIVE}")
print(f"Completed checkpoints: {list(checkpoints.keys()) or 'none (fresh run)'}")

In [ ]:
# Cell 3 — Clone repo and install
!git clone https://github.com/Satyam810/LexAi.git lexai
%cd lexai
!pip install -q -r requirements_colab.txt
print("Ready.")

In [ ]:
# Cell 4 — Upload cases.json
from google.colab import files
import os, json

os.makedirs('data/processed', exist_ok=True)
print("Upload your local data/processed/cases.json:")
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/processed/{name}', 'wb') as f:
        f.write(content)

with open('data/processed/cases.json') as f:
    cases = json.load(f)
print(f"Loaded {len(cases)} cases.")

In [ ]:
# Cell 5 — Batch embeddings (GPU) — WITH CHECKPOINT
import json, numpy as np, torch, shutil
from sentence_transformers import SentenceTransformer
from pathlib import Path

if checkpoints.get("embeddings"):
    print("Embeddings already done. Loading from disk.")
    embeddings = np.load('data/processed/embeddings.npy')
    print(f"Shape: {embeddings.shape}")
else:
    with open('data/processed/cases.json') as f:
        cases = json.load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Embedding on: {device}")
    model = SentenceTransformer("nlpaueb/legal-bert-base-uncased", device=device)
    texts = [" ".join(c["text"].split()[:512]) for c in cases]

    embeddings = model.encode(
        texts, batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    Path('data/processed').mkdir(parents=True, exist_ok=True)
    np.save('data/processed/embeddings.npy', embeddings)
    shutil.copy('data/processed/embeddings.npy', f'{GDRIVE}/embeddings.npy')
    save_checkpoint("embeddings")
    print(f"Shape: {embeddings.shape} | Saved to Drive.")

In [ ]:
# Cell 6 — FAISS index — WITH CHECKPOINT
import faiss, numpy as np, shutil

if checkpoints.get("faiss"):
    print("FAISS already done. Skipping.")
else:
    embeddings = np.load('data/processed/embeddings.npy').astype("float32")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, 'data/processed/faiss.index')
    shutil.copy('data/processed/faiss.index', f'{GDRIVE}/faiss.index')
    save_checkpoint("faiss")
    print(f"FAISS index: {index.ntotal} vectors. Saved to Drive.")

In [ ]:
# Cell 7 — HDBSCAN vs KMeans — WITH CHECKPOINT
import numpy as np, json, shutil
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import hdbscan

if checkpoints.get("clustering"):
    print("Clustering already done. Loading from disk.")
    final_labels = np.load('data/processed/cluster_labels.npy')
    with open('data/processed/eval_metrics.json') as f:
        metrics = json.load(f)
    print(f"Winner: {metrics['winner_algorithm']}, k={metrics['n_clusters']}")
else:
    embeddings = np.load('data/processed/embeddings.npy')
    sample = min(500, len(embeddings))

    print("=== KMeans sweep ===")
    km_results = []
    for k in range(5, 21):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(embeddings)
        sil = silhouette_score(embeddings, labels, sample_size=sample, random_state=42)
        db  = davies_bouldin_score(embeddings, labels)
        km_results.append({"k": k, "sil": sil, "db": db, "labels": labels})
        print(f"  k={k:2d}  sil={sil:.4f}  db={db:.4f}")
    best_km = max(km_results, key=lambda x: x["sil"])

    print("\n=== HDBSCAN ===")
    hdb = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
    hdb_labels = hdb.fit_predict(embeddings)
    n_hdb  = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    valid  = hdb_labels != -1
    hdb_sil = (silhouette_score(embeddings[valid], hdb_labels[valid],
                                sample_size=min(500, valid.sum()), random_state=42)
               if valid.sum() > 10 and n_hdb > 1 else 0.0)
    hdb_db  = (davies_bouldin_score(embeddings[valid], hdb_labels[valid])
               if valid.sum() > 10 and n_hdb > 1 else 999.0)
    print(f"HDBSCAN: {n_hdb} clusters | sil={hdb_sil:.4f} | db={hdb_db:.4f}")

    if best_km["sil"] >= hdb_sil:
        winner = "kmeans"; final_labels = best_km["labels"]
        final_sil = best_km["sil"]; final_db = best_km["db"]; final_k = best_km["k"]
    else:
        winner = "hdbscan"; final_labels = hdb_labels
        final_sil = hdb_sil; final_db = hdb_db; final_k = n_hdb

    print(f"\nWINNER: {winner.upper()} | k={final_k} | sil={final_sil:.4f}")

    np.save('data/processed/cluster_labels.npy', final_labels)
    shutil.copy('data/processed/cluster_labels.npy', f'{GDRIVE}/cluster_labels.npy')

    metrics = {
        "winner_algorithm": winner, "n_clusters": int(final_k),
        "silhouette_score": round(float(final_sil), 4),
        "davies_bouldin_score": round(float(final_db), 4),
        "kmeans_best_k": int(best_km["k"]),
        "kmeans_best_silhouette": round(float(best_km["sil"]), 4),
        "hdbscan_n_clusters": int(n_hdb),
        "hdbscan_silhouette": round(float(hdb_sil), 4),
        "total_cases": len(embeddings),
    }
    with open('data/processed/eval_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    shutil.copy('data/processed/eval_metrics.json', f'{GDRIVE}/eval_metrics.json')
    save_checkpoint("clustering")
    print(f"\n*** SAVE FOR RESUME: {winner.upper()} k={final_k} sil={final_sil:.4f} ***")

In [ ]:
# Cell 8 — UMAP 2D — WITH CHECKPOINT + RAM GUARD
import umap, numpy as np, shutil

if checkpoints.get("umap"):
    print("UMAP already done. Skipping.")
else:
    embeddings = np.load('data/processed/embeddings.npy')
    umap_emb = embeddings
    if len(embeddings) > 800:
        print(f"WARNING: {len(embeddings)} cases -- subsampling 800 for UMAP (RAM guard).")
        import numpy.random as rng
        idx = rng.choice(len(embeddings), 800, replace=False)
        umap_emb = embeddings[idx]

    print(f"UMAP running on {len(umap_emb)} cases (~3 min)...")
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
    coords = reducer.fit_transform(umap_emb)

    np.save('data/processed/coords_2d.npy', coords)
    shutil.copy('data/processed/coords_2d.npy', f'{GDRIVE}/coords_2d.npy')
    save_checkpoint("umap")
    print(f"UMAP done. Shape: {coords.shape}. Saved to Drive.")

In [ ]:
# Cell 9 — BERTopic cluster labels — WITH CHECKPOINT
import json, numpy as np, shutil
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict

if checkpoints.get("bertopic"):
    print("BERTopic already done. Skipping.")
else:
    with open('data/processed/cases.json') as f:
        cases = json.load(f)
    labels = np.load('data/processed/cluster_labels.npy')
    texts  = [c["text"][:1000] for c in cases]
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1,2), max_features=5000)
    cluster_texts = defaultdict(list)
    for text, label in zip(texts, labels):
        if label != -1:
            cluster_texts[int(label)].append(text)

    cluster_topics = {}
    for cid, ctexts in cluster_texts.items():
        if len(ctexts) < 3:
            cluster_topics[cid] = f"Cluster {cid}"
            continue
        try:
            m = BERTopic(vectorizer_model=vectorizer, nr_topics=1, verbose=False)
            m.fit_transform(ctexts)
            info = m.get_topic(0)
            cluster_topics[cid] = (
                " | ".join(w for w, _ in info[:4]) if info else f"Cluster {cid}"
            )
        except:
            cluster_topics[cid] = f"Cluster {cid}"

    with open('data/processed/cluster_topics.json', 'w') as f:
        json.dump(cluster_topics, f, indent=2)
    shutil.copy('data/processed/cluster_topics.json', f'{GDRIVE}/cluster_topics.json')
    save_checkpoint("bertopic")
    print(f"Labeled {len(cluster_topics)} clusters.")

In [ ]:
# Cell 10 — Gap detection — WITH CHECKPOINT
import json, numpy as np, shutil
from collections import defaultdict

if checkpoints.get("gaps"):
    print("Gap detection already done. Skipping.")
else:
    with open('data/processed/cases.json') as f:
        cases = json.load(f)
    labels = np.load('data/processed/cluster_labels.npy')

    groups = defaultdict(list)
    for case, label in zip(cases, labels):
        if int(label) != -1:
            groups[int(label)].append(case)

    gaps = []
    for cid, members in groups.items():
        verdict_counts = defaultdict(int)
        for m in members:
            verdict_counts[m["verdict"]] += 1
        
        granted = verdict_counts.get("bail_granted", 0) + verdict_counts.get("acquitted", 0)
        rejected = verdict_counts.get("bail_rejected", 0) + verdict_counts.get("convicted", 0)
        
        if not granted or not rejected:
            continue
        
        total = len(members)
        score = round(min(granted, rejected) / total, 3)
        
        all_sections = []
        for m in members:
            all_sections.extend(m["ipc_sections"])
        common_sections = sorted(
            set(all_sections), key=lambda s: -all_sections.count(s)
        )[:5]
        
        gaps.append({
            "cluster_id": cid, "total_cases": total,
            "positive_outcome_count": granted,
            "negative_outcome_count": rejected,
            "convicted_count": verdict_counts.get("convicted", 0),
            "acquitted_count": verdict_counts.get("acquitted", 0),
            "bail_granted_count": verdict_counts.get("bail_granted", 0),
            "bail_rejected_count": verdict_counts.get("bail_rejected", 0),
            "inconsistency_score": score,
            "common_ipc_sections": common_sections,
            "dominant_case_type": max(
                set(m["case_type"] for m in members),
                key=lambda t: sum(1 for m in members if m["case_type"] == t)
            ),
            "courts_involved": list(set(m["court"] for m in members))[:5],
        })

    gaps = sorted(gaps, key=lambda x: -x["inconsistency_score"])
    with open('data/processed/gaps.json', 'w') as f:
        json.dump(gaps, f, indent=2)
    shutil.copy('data/processed/gaps.json', f'{GDRIVE}/gaps.json')
    save_checkpoint("gaps")
    print(f"Gaps found: {len(gaps)}")
    if gaps:
        print(f"\n*** SAVE FOR RESUME: {len(gaps)} inconsistent clusters, "
              f"top={gaps[0]['inconsistency_score']:.0%} ***")

In [ ]:
# Cell 11 — Download all 7 outputs
from google.colab import files
import os

outputs = [
    'data/processed/embeddings.npy',
    'data/processed/faiss.index',
    'data/processed/cluster_labels.npy',
    'data/processed/cluster_topics.json',
    'data/processed/coords_2d.npy',
    'data/processed/gaps.json',
    'data/processed/eval_metrics.json',
]

print("Downloading outputs...")
for f_path in outputs:
    if os.path.exists(f_path):
        files.download(f_path)
        print(f"  OK: {f_path}")
    else:
        print(f"  MISSING: {f_path}")

print(f"\nCheckpoint status: {load_checkpoints()}")
print("\nCopy all downloaded files into your local data/processed/ folder.")
print("Then run: python verify_outputs.py")
print("Then run: python src/eval_pipeline.py   <-- NEW Phase 5.5")